In [ ]:
# === PARAMETERS ===
MODEL_TYPE = "COORD" 
LOG_FILE = "value_training_log.txt"
NAME_ID = "val_cent_max_v1"

# Grid / Agents
WIDTH = 9
HEIGHT = 9
NUM_AGENTS = 4
SPAWN_PROB = 0.05
DESPAWN_PROB = 0.09

# Model Config (High Capacity)
MLP_HIDDEN_DIM = 1024
MLP_NUM_LAYERS = 4
CNN_CONV_CHANNELS = [64, 64, 64] 
CNN_HEAD_HIDDEN_DIM = 512
CNN_HEAD_NUM_LAYERS = 3
CNN_KERNEL_SIZE = 3

# Value Learning Config
LR = 0.00005
BATCH_SIZE = 512
DISCOUNT = 0.99
TARGET_UPDATE_FREQ = 1000
BUFFER_SIZE = 50000
SEED = 42
MAX_STEPS = 10000000  # Effectively Infinite (Killed by SLURM)
EVAL_FREQ = 10000     # 10000
EVAL_STEPS = 2000

# Epsilon Greedy
EPSILON = 0.1

SAVE_FREQ = 50000

In [9]:
OUT_FOLDER = f"value_centralized_{MODEL_TYPE}_lr{LR}_{NAME_ID}"

In [10]:
from pathlib import Path
import sys
import numpy as np
import torch
import random
import ast
import math
from tqdm import tqdm

sys.path.append("../") 

from tadd_helpers.env_functions import init_empty_state, spawn_apples, despawn_apples, State
from tadd_helpers.eval_functions import nearest_policy, random_policy
from models.value_mlp import ValueMLPCentralized
from models.value_cnn_new import ValueCNNCentralizedStandard, ValueCNNCoord
from orchard.environment import MoveAction

# Parse string params if needed
if isinstance(CNN_CONV_CHANNELS, str):
    CNN_CONV_CHANNELS = ast.literal_eval(CNN_CONV_CHANNELS)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

OUT_FOLDER = Path(OUT_FOLDER)
OUT_FOLDER.mkdir(parents=True, exist_ok=True)
LOG_FILE = OUT_FOLDER / LOG_FILE

In [11]:
from models.value_cnn_new import ReplayBuffer

# --- INITIALIZATION --+
if MODEL_TYPE == "MLP":
    model = ValueMLPCentralized(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=MLP_HIDDEN_DIM, num_layers=MLP_NUM_LAYERS
    )
elif MODEL_TYPE == "CNN":
    model = ValueCNNCentralizedStandard(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=CNN_HEAD_HIDDEN_DIM, num_layers=CNN_HEAD_NUM_LAYERS, 
        conv_channels=CNN_CONV_CHANNELS, kernel_size=CNN_KERNEL_SIZE
    )
elif MODEL_TYPE == "COORD":
    model = ValueCNNCoord(
        height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
        hidden_dim=CNN_HEAD_HIDDEN_DIM, 
        num_layers=CNN_HEAD_NUM_LAYERS, 
        conv_channels=CNN_CONV_CHANNELS, 
        kernel_size=CNN_KERNEL_SIZE
    )
else:
    raise ValueError(f"Invalid MODEL_TYPE: {MODEL_TYPE}")

# Force Expand Buffer for Long Run
model.memory = ReplayBuffer(BUFFER_SIZE)

print(f"Initialized Centralized {MODEL_TYPE}. Buffer: 1M. LR: {LR}")

Initialized Centralized COORD. Buffer: 1M. LR: 5e-05


In [12]:
import psutil
import os
def get_ram_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024  # Returns MB

In [13]:
def get_best_action_lookahead(model, state: State, agent_idx: int):
    """
    Simulates 4 moves. Pick max V(s_mid).
    Includes RANDOM TIE-BREAKING to prevent directional bias.
    """
    agent_pos = state.agent_position(agent_idx)
    
    # Store tuples of (value, action)
    candidates = []
    
    for action in MoveAction:
        new_pos = np.clip(agent_pos + action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
        
        s_mid = state.copy()
        s_mid.set_agent_position(agent_idx, new_pos)
        
        # Query Model
        val = model.get_value(s_mid)
        candidates.append((val, action))
    
    # Find max value
    max_val = max(c[0] for c in candidates)
    
    # Filter all actions that match max_val (within float tolerance)
    best_actions = [act for (val, act) in candidates if abs(val - max_val) < 1e-5]
    
    # Random tie-break
    return random.choice(best_actions)

def run_evaluation_episode(policy_func, steps=500):
    s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
    total_reward = 0.0
    
    for _ in range(steps):
        active_idx = s.get_random_agent_id()
        action = policy_func(s, active_idx)
        
        new_pos = np.clip(s.agent_position(active_idx) + action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
        s.set_agent_position(active_idx, new_pos)
        
        step_reward = 0.0
        if s.apples[tuple(new_pos)] > 0:
            s.remove_apple_at(new_pos)
            step_reward = 1.0
            
        total_reward += step_reward
        spawn_apples(s, SPAWN_PROB)
        despawn_apples(s, DESPAWN_PROB)
        
    return total_reward / steps

def learned_policy_wrapper(s, idx):
    return get_best_action_lookahead(model, s, idx)

In [14]:
loss_history = []
val_history = []

model.memory.memory.clear()
model.update_target_net()

s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)

with open(LOG_FILE, "a") as f:
    f.write(f"\n=== STARTING CENTRALIZED MAX PERF: {MODEL_TYPE} ===\n")
    f.write("Step   | Loss    | Avg_V   | Max_V   | Delta | Act% | AvgR_Learn | AvgR_Near | Status\n")

pbar = tqdm(range(MAX_STEPS))
for step in pbar:
    
    # --- 1. Action Selection ---
    active_agent_idx = s.get_random_agent_id()
    current_val_pred = 0.0
    
    if random.random() < EPSILON:
        move_action = random_policy(s, active_agent_idx)
        with torch.no_grad():
            current_val_pred = model.get_value(s)
    else:
        move_action = get_best_action_lookahead(model, s, active_agent_idx)
        with torch.no_grad():
            current_val_pred = model.get_value(s)
            
    val_history.append(current_val_pred)
    
    # --- 2. TRANSITION A: Move (s -> s_mid) ---
    new_pos = np.clip(s.agent_position(active_agent_idx) + move_action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
    s_mid = s.copy()
    s_mid.set_agent_position(active_agent_idx, new_pos)
    
    model.add_experience(s, s_mid, 0.0)
    
    # --- 3. TRANSITION B: Pick/Spawn (s_mid -> s_next) ---
    s_next = s_mid.copy()
    reward = 0.0
    if s_next.apples[tuple(new_pos)] > 0:
        s_next.remove_apple_at(new_pos)
        reward = 1.0 
        
    spawn_apples(s_next, SPAWN_PROB)
    despawn_apples(s_next, DESPAWN_PROB)
    
    model.add_experience(s_mid, s_next, reward)
    
    # --- 4. Train with Gradient Clipping ---
    if len(model.memory) >= BATCH_SIZE:
        loss = model.train_batch(BATCH_SIZE)
        if loss is not None: loss_history.append(loss)
    else:
        loss = 0.0

    if step % TARGET_UPDATE_FREQ == 0:
        model.update_target_net()
        
    s = s_next
    
    # --- Save Models ---
    if step % SAVE_FREQ == 0:
        model_path = OUT_FOLDER / "models" / f"model_step{step}.pt"
        model_path.parent.mkdir(parents=True, exist_ok=True)
        torch.save(model.state_dict(), model_path)
    
    # --- 5. Eval ---
    if step % EVAL_FREQ == 0 and step > 0:
        recent_vals = val_history[-EVAL_FREQ:]
        avg_v = np.mean(recent_vals)
        max_v = np.max(recent_vals)
        
        # === ADD THIS: Quick diagnostic (fast, ~100 samples) ===
        deltas = []
        correct_actions = 0
        n_test = 100
        for _ in range(n_test):
            s_test = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
            s_test.apples[:] = 0
            pos = s_test.agent_position(0)
            
            # Delta test
            s_on = s_test.copy()
            s_on.apples[pos[0], pos[1]] = 1
            deltas.append(model.get_value(s_on) - model.get_value(s_test))
            
            # Action test
            apple_r, apple_c = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
            if (apple_r, apple_c) != tuple(pos):
                s_test.apples[apple_r, apple_c] = 1
                best_val, best_act = -1e9, None
                for act in MoveAction:
                    npos = np.clip(pos + act.vector, [0,0], [HEIGHT-1, WIDTH-1])
                    st = s_test.copy()
                    st.set_agent_position(0, npos)
                    v = model.get_value(st)
                    if v > best_val:
                        best_val, best_act = v, act
                new_p = np.clip(pos + best_act.vector, [0,0], [HEIGHT-1, WIDTH-1])
                old_d = abs(pos[0]-apple_r) + abs(pos[1]-apple_c)
                new_d = abs(new_p[0]-apple_r) + abs(new_p[1]-apple_c)
                if new_d < old_d:
                    correct_actions += 1
        
        delta_mean = np.mean(deltas)
        act_acc = correct_actions
        # === END ADDITION ===
        
        r_learned = run_evaluation_episode(learned_policy_wrapper, EVAL_STEPS)
        r_nearest = run_evaluation_episode(nearest_policy, EVAL_STEPS)
        
        status = ""
        if r_learned > r_nearest:
            status = "BEAT_NEAREST"
        ram_mb = get_ram_usage()
        with open(LOG_FILE, "a") as f:
            # Updated format with delta and action accuracy
            f.write(f"{step:<6} | {loss:.5f} | {avg_v:6.2f}  | {max_v:6.2f}  | {delta_mean:5.2f} | {act_acc:3d}% | {r_learned:.4f}     | {r_nearest:.4f}    | {status} | RAM:{ram_mb:.0f}MB\n")
            
        pbar.set_description(f"L:{loss:.4f} | V:{avg_v:.1f} | Δ:{delta_mean:.2f} | R:{r_learned:.2f}")

L:0.0000 | V:0.0 | Δ:-0.00 | R:0.05:   0%|          | 200/10000000 [00:20<288:09:28,  9.64it/s] 


KeyboardInterrupt: 